# P3 — RoBERTa-History (Nautilus)

Plain supervised roberta-base classifier: history + speaker/emotion tags +
next-speaker identity -> emotion. Never sees the target utterance's text.

**GPU requirement (Nautilus):** Any single GPU 16GB+ -- full config, larger real batch size (no gradient accumulation needed).

**Before running:** set `DATA_PATH` to your canonical-split IEMOCAP pkl.
Assumes this notebook sits inside `p3_roberta/`, with `../p0_evaluation/`
as a sibling for the final scoring cell.


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
DATA_PATH = "/path/to/iemocap.pkl"  # <-- EDIT THIS
import os
assert os.path.exists(DATA_PATH), f"DATA_PATH does not exist: {DATA_PATH} -- edit the cell above"
os.environ["DATA_PATH"] = DATA_PATH


In [ ]:
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg.update({"batch_size": 16, "gradient_accumulation_steps": 1})  # bigger GPU -> bigger real batch, no accumulation needed
with open("config_nautilus.yaml", "w") as f:
    yaml.safe_dump(cfg, f)
print(cfg)


In [ ]:
!python run.py --mode train --data_path "$DATA_PATH" --config config_nautilus.yaml


In [ ]:
!python run.py --mode eval --data_path "$DATA_PATH" --config config_nautilus.yaml \
    --model_path outputs/best_model --save_path outputs/predictions.json


In [ ]:
!cd ../p0_evaluation && python run_eval.py --preds ../p3_roberta/outputs/predictions.json --out ../p3_roberta/outputs/p0_result.json
import json
print(json.dumps(json.load(open("outputs/p0_result.json"))["metrics"], indent=2))
